# Can we predict late delivery?
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, choose by mean CV ROC AUC, score the chosen model once on the holdout).

Feature decision: `claim_paid_eur` is compensation paid on a claim settled for the consignment, so it is recorded after the outcome (mean 78 EUR for late vs 1 EUR for on-time). It is **excluded** as leakage. `consignment` (identifier) and `dispatched_on` (split column) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['t']=pd.to_datetime(df['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train=df.drop(index=hold_idx).sort_values('t',kind='stable')   # index keeps 0-based row positions
hold=df.loc[hold_idx]
print(len(train),len(hold),train.t.max(),hold.t.min(),train.late_delivery.mean(),hold.late_delivery.mean())

In [ ]:
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT
y_col='late_delivery'
def pre(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
# all candidates named before any is scored
candidates={
 'baseline_prior':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'rf_depth6':Pipeline([('p',pre(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small':Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
X=train[FEATURES]; y=train[y_col]
tscv=TimeSeriesSplit(n_splits=5)
folds=[(tr,va) for tr,va in tscv.split(X)]
cv_folds=[[train.index[tr].tolist(),train.index[va].tolist()] for tr,va in folds]
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(X.iloc[tr],y.iloc[tr])   # preprocessing fitted inside the fold
        s.append(float(roc_auc_score(y.iloc[va],m.predict_proba(X.iloc[va])[:,1])))
    cv_scores[name]=s
res=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(res.sort_values('mean',ascending=False))
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k])); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X,y)
auc=float(roc_auc_score(hold[y_col],final.predict_proba(hold[FEATURES])[:,1]))
print(f'Holdout ROC AUC of {chosen}: {auc:.4f}')
evidence={'features':FEATURES,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in hold_idx],'holdout_scores':{chosen:{'roc_auc':auc}}}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the printed CV table and holdout ROC AUC above. The holdout score was computed once, for the model chosen on CV alone. `claim_paid_eur` would have given a near-perfect but meaningless score because it is only known after delivery.